# CPI Prediction — Analysis

Loads `data/processed/monthly_panel.csv` from `01_data_collection.ipynb` and runs:
1. Descriptive analysis and correlation table
2. Category-level OLS regressions (macro/surveys, prediction markets, sentiment)
3. Unified LASSO (all features compete)
4. Model comparison plots and summary table

In [ ]:
import os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import statsmodels.api as sm
from sklearn.linear_model import LassoCV
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import LeaveOneOut
from sklearn.metrics import r2_score, mean_squared_error

warnings.filterwarnings('ignore')
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')

os.makedirs('output/figures', exist_ok=True)
os.makedirs('output/tables',  exist_ok=True)

%matplotlib inline
plt.rcParams.update({'figure.dpi':150, 'font.size':11, 'axes.spines.top':False, 'axes.spines.right':False})

panel = pd.read_csv('data/processed/monthly_panel.csv', index_col=0, parse_dates=True)
panel.index = pd.to_datetime(panel.index) + pd.offsets.MonthEnd(0)

MACRO_VARS     = ['michigan_inflation','cleveland_inflation','spf_cpi_forecast',
                  'unemployment','ppi','oil_wti','pce_yoy']
MARKET_VARS    = ['kalshi_implied_cpi']
SENTIMENT_VARS = ['gdelt_tone','google_trends_inflation']
ALL_VARS       = MACRO_VARS + MARKET_VARS + SENTIMENT_VARS

CATEGORY_MAP = {v:'Macro/Survey' for v in MACRO_VARS}
CATEGORY_MAP.update({v:'Prediction Market' for v in MARKET_VARS})
CATEGORY_MAP.update({v:'Sentiment' for v in SENTIMENT_VARS})

CAT_COLORS = {'Macro/Survey':'#2196F3','Prediction Market':'#FF5722','Sentiment':'#4CAF50'}

print(f"Panel loaded: {panel.shape[0]} rows x {panel.shape[1]} cols")
print(f"Date range: {panel.index[0].date()} to {panel.index[-1].date()}")
panel.head(3)

## 1. Descriptive Analysis

### 1a. Time series: Actual CPI vs. key predictors

In [ ]:
fig, ax = plt.subplots(figsize=(12,5))

ax.plot(panel.index, panel['cpi_yoy'], color='black', linewidth=2.2, label='Actual CPI YoY', zorder=5)
ax.plot(panel.index, panel['michigan_inflation'], color='#2196F3', linewidth=1.4,
        linestyle='--', label='Michigan Survey (1yr exp.)', alpha=0.85)
ax.plot(panel.index, panel['spf_cpi_forecast'], color='#1565C0', linewidth=1.4,
        linestyle=':', label='SPF Forecast', alpha=0.85)

if panel['kalshi_implied_cpi'].notna().sum() > 3:
    ax.plot(panel.index, panel['kalshi_implied_cpi'], color='#FF5722', linewidth=1.6,
            label='Kalshi Implied CPI', alpha=0.9)
else:
    ax.annotate('Kalshi: add credentials\nto 01_data_collection.ipynb',
                xy=(0.72,0.88), xycoords='axes fraction', color='#FF5722',
                fontsize=9, style='italic')

ax.fill_between(panel.index,
                panel['cpi_yoy'].min() - 0.5,
                panel['cpi_yoy'].max() + 0.5,
                where=(panel['cpi_yoy'] > 5), alpha=0.06, color='red', label='CPI > 5%')

ax.set_xlabel('')
ax.set_ylabel('Percent (%)')
ax.set_title('CPI YoY vs. Prediction Sources (Jan 2022 – Apr 2026)', fontweight='bold')
ax.legend(loc='upper right', fontsize=9)
ax.xaxis.set_major_formatter(plt.matplotlib.dates.DateFormatter('%b %Y'))
ax.xaxis.set_major_locator(plt.matplotlib.dates.MonthLocator(interval=4))
plt.xticks(rotation=40, ha='right')
plt.tight_layout()
plt.savefig('output/figures/fig1_timeseries.png', bbox_inches='tight')
plt.show()
print("Saved output/figures/fig1_timeseries.png")

### 1b. Correlation with next-month CPI

In [ ]:
cor_data = panel[['cpi_yoy_next'] + ALL_VARS].dropna()
cor_vals = cor_data.corr()['cpi_yoy_next'].drop('cpi_yoy_next')

cor_df = pd.DataFrame({
    'Variable':    cor_vals.index,
    'Category':    [CATEGORY_MAP[v] for v in cor_vals.index],
    'Correlation': cor_vals.values.round(3)
}).sort_values('Correlation', key=abs, ascending=False).reset_index(drop=True)

cor_df.to_csv('output/tables/correlation_table.csv', index=False)

fig, ax = plt.subplots(figsize=(9,5))
colors = [CAT_COLORS[c] for c in cor_df['Category']]
bars = ax.barh(cor_df['Variable'], cor_df['Correlation'], color=colors, edgecolor='white')
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('Correlation with CPI YoY (t+1)')
ax.set_title('Predictor Correlations with Next-Month CPI', fontweight='bold')
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=v, label=k) for k,v in CAT_COLORS.items()]
ax.legend(handles=legend_elements, loc='lower right', fontsize=9)
plt.tight_layout()
plt.savefig('output/figures/fig1b_correlations.png', bbox_inches='tight')
plt.show()
print("Saved output/figures/fig1b_correlations.png")
display(cor_df)

## 2. Category-Level OLS

Three separate OLS regressions, one per category. Predictors standardized (z-scored) so coefficients are comparable. Reports R², adjusted R², and RMSE.

In [ ]:
scaler = StandardScaler()
panel_std = panel.copy()
panel_std[ALL_VARS] = scaler.fit_transform(panel[ALL_VARS])

def run_ols(vars_list, label):
    data = panel_std[['cpi_yoy_next'] + vars_list].dropna()
    y = data['cpi_yoy_next'].values
    X = sm.add_constant(data[vars_list].values)
    fit = sm.OLS(y, X).fit()
    preds = fit.predict(X)
    rmse  = np.sqrt(mean_squared_error(y, preds))
    return fit, {'Model': label, 'N': int(fit.nobs),
                 'R2': round(fit.rsquared, 3),
                 'Adj_R2': round(fit.rsquared_adj, 3),
                 'RMSE': round(rmse, 3)}

fit_macro,    row_macro    = run_ols(MACRO_VARS,     'Macro/Surveys OLS')
fit_market,   row_market   = run_ols(MARKET_VARS,    'Prediction Markets OLS')
fit_sentiment,row_sentiment= run_ols(SENTIMENT_VARS, 'Sentiment OLS')

ols_results = pd.DataFrame([row_macro, row_market, row_sentiment])
ols_results.to_csv('output/tables/ols_results.csv', index=False)
print("OLS Summary:")
display(ols_results)

### OLS Coefficient Plots

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 5), sharey=False)
models  = [fit_macro,    fit_market,   fit_sentiment]
labels  = ['Macro/Surveys', 'Prediction\nMarkets', 'Sentiment']
varsets = [MACRO_VARS,   MARKET_VARS,  SENTIMENT_VARS]
colors  = ['#2196F3',    '#FF5722',    '#4CAF50']

for ax, fit, label, varset, color in zip(axes, models, labels, varsets, colors):
    coefs  = fit.params[1:]
    ci_low = fit.conf_int().iloc[1:, 0]
    ci_hi  = fit.conf_int().iloc[1:, 1]
    yerr   = np.array([coefs - ci_low, ci_hi - coefs])
    ax.barh(varset, coefs, xerr=yerr, color=color, alpha=0.8,
            error_kw={'ecolor':'gray','capsize':3})
    ax.axvline(0, color='black', linewidth=0.8, linestyle='--')
    ax.set_title(f'{label}\nR²={fit.rsquared:.3f}', fontweight='bold')
    ax.set_xlabel('Std. Coefficient')

plt.suptitle('OLS Coefficients by Category (with 95% CI)', fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('output/figures/fig2_ols_coefs.png', bbox_inches='tight')
plt.show()
print("Saved output/figures/fig2_ols_coefs.png")

## 3. Unified LASSO

All features from all three categories compete in a single LASSO regression.
Lambda selected via Leave-One-Out cross-validation.

The features that survive (non-zero coefficients) indicate which category carries the most independent information about future CPI.

In [ ]:
lasso_data = panel[['cpi_yoy_next'] + ALL_VARS].dropna()
X_raw = lasso_data[ALL_VARS].values
y     = lasso_data['cpi_yoy_next'].values
n     = len(y)

scl   = StandardScaler()
X     = scl.fit_transform(X_raw)

loo   = LeaveOneOut()
lasso = LassoCV(cv=loo, max_iter=50000, random_state=42, n_alphas=200)
lasso.fit(X, y)

print(f"Optimal alpha (lambda): {lasso.alpha_:.4f}")
print(f"N observations:         {n}")
print(f"N features:             {len(ALL_VARS)}")

lasso_preds = lasso.predict(X)
lasso_r2    = r2_score(y, lasso_preds)
lasso_rmse  = np.sqrt(mean_squared_error(y, lasso_preds))
print(f"\nIn-sample R²:  {lasso_r2:.3f}")
print(f"In-sample RMSE: {lasso_rmse:.3f}")

In [ ]:
coef_df = pd.DataFrame({
    'Variable':    ALL_VARS,
    'Coefficient': lasso.coef_,
    'Category':    [CATEGORY_MAP[v] for v in ALL_VARS]
}).query('Coefficient != 0').sort_values('Coefficient', key=abs, ascending=False)

coef_df.to_csv('output/tables/lasso_results.csv', index=False)
print(f"{len(coef_df)} non-zero coefficients:")
display(coef_df)

In [ ]:
if len(coef_df) > 0:
    fig, ax = plt.subplots(figsize=(8, max(4, len(coef_df)*0.55)))
    colors_bar = [CAT_COLORS[c] for c in coef_df['Category']]
    ax.barh(coef_df['Variable'], coef_df['Coefficient'], color=colors_bar, edgecolor='white')
    ax.axvline(0, color='black', linewidth=0.8)
    ax.set_xlabel('Standardized Coefficient')
    ax.set_title(f'LASSO: Non-Zero Coefficients by Category\n'
                 f'α={lasso.alpha_:.4f}  |  R²={lasso_r2:.3f}  |  RMSE={lasso_rmse:.3f}',
                 fontweight='bold')
    legend_elements = [Patch(facecolor=v, label=k) for k,v in CAT_COLORS.items()]
    ax.legend(handles=legend_elements, fontsize=9)
    plt.tight_layout()
    plt.savefig('output/figures/fig3_lasso_coefs.png', bbox_inches='tight')
    plt.show()
    print("Saved output/figures/fig3_lasso_coefs.png")
else:
    print("All coefficients zeroed out — try a smaller alpha.")
    print("Current alpha:", lasso.alpha_)

## 4. Model Comparison

In [ ]:
lasso_metrics = pd.DataFrame([{
    'Model': 'Unified LASSO',
    'N':      n,
    'R2':     round(lasso_r2, 3),
    'Adj_R2': round(1 - (1-lasso_r2)*(n-1)/(n-len(coef_df)-1), 3) if len(coef_df) > 0 else np.nan,
    'RMSE':   round(lasso_rmse, 3)
}])

comparison = pd.concat([ols_results, lasso_metrics], ignore_index=True)
comparison.to_csv('output/tables/model_comparison.csv', index=False)
display(comparison)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

model_colors = ['#2196F3','#FF5722','#4CAF50','#9C27B0']
models_list  = comparison['Model'].tolist()
r2_vals      = comparison['R2'].tolist()
rmse_vals    = comparison['RMSE'].tolist()

# R² bar chart
ax = axes[0]
bars = ax.bar(range(len(models_list)), r2_vals, color=model_colors, edgecolor='white', width=0.6)
ax.set_xticks(range(len(models_list)))
ax.set_xticklabels([m.replace(' OLS','\nOLS').replace('Unified','Unified\n') for m in models_list],
                   fontsize=9)
ax.set_ylabel('R²')
ax.set_title('In-Sample R² by Model', fontweight='bold')
ax.set_ylim(0, min(1.1, max(r2_vals)*1.3 + 0.05))
for bar, val in zip(bars, r2_vals):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01,
            f'{val:.3f}', ha='center', va='bottom', fontsize=10, fontweight='bold')

# RMSE bar chart
ax = axes[1]
bars = ax.bar(range(len(models_list)), rmse_vals, color=model_colors, edgecolor='white', width=0.6)
ax.set_xticks(range(len(models_list)))
ax.set_xticklabels([m.replace(' OLS','\nOLS').replace('Unified','Unified\n') for m in models_list],
                   fontsize=9)
ax.set_ylabel('RMSE (percentage points)')
ax.set_title('In-Sample RMSE by Model\n(lower is better)', fontweight='bold')
for bar, val in zip(bars, rmse_vals):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005,
            f'{val:.3f}', ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.suptitle('Model Comparison: Predicting Next-Month CPI YoY (Jan 2022–Mar 2026)',
             fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('output/figures/fig4_model_comparison.png', bbox_inches='tight')
plt.show()
print("Saved output/figures/fig4_model_comparison.png")

## 5. LASSO Coefficient Path (Regularization Path)

In [ ]:
from sklearn.linear_model import lasso_path

alphas, coef_paths, _ = lasso_path(X, y, alphas=np.logspace(-3, 1, 200))

fig, ax = plt.subplots(figsize=(10, 5))
for i, var in enumerate(ALL_VARS):
    cat   = CATEGORY_MAP[var]
    color = CAT_COLORS[cat]
    ls    = '-' if cat=='Macro/Survey' else ('--' if cat=='Prediction Market' else ':')
    ax.plot(np.log10(alphas), coef_paths[i], color=color, linestyle=ls, linewidth=1.3,
            label=var if abs(coef_paths[i, -1]) > 0.01 or abs(coef_paths[i, 0]) > 0.05 else '_')

ax.axvline(np.log10(lasso.alpha_), color='black', linestyle='--', linewidth=1.2,
           label=f'Chosen α={lasso.alpha_:.4f}')
ax.set_xlabel('log₁₀(α)')
ax.set_ylabel('Standardized Coefficient')
ax.set_title('LASSO Regularization Path', fontweight='bold')
ax.legend(fontsize=7, ncol=2, loc='upper right')

legend_elements = [Patch(facecolor=v, label=k) for k,v in CAT_COLORS.items()]
leg2 = ax.legend(handles=legend_elements, fontsize=9, loc='upper left')
ax.add_artist(leg2)
plt.tight_layout()
plt.savefig('output/figures/fig5_lasso_path.png', bbox_inches='tight')
plt.show()
print("Saved output/figures/fig5_lasso_path.png")

## Summary

| Output | Location |
|--------|----------|
| Time series plot | `output/figures/fig1_timeseries.png` |
| Correlation bar chart | `output/figures/fig1b_correlations.png` |
| OLS coefficient plots | `output/figures/fig2_ols_coefs.png` |
| LASSO coefficient bar | `output/figures/fig3_lasso_coefs.png` |
| Model R²/RMSE comparison | `output/figures/fig4_model_comparison.png` |
| LASSO regularization path | `output/figures/fig5_lasso_path.png` |
| Correlation table | `output/tables/correlation_table.csv` |
| OLS results | `output/tables/ols_results.csv` |
| LASSO results | `output/tables/lasso_results.csv` |
| Model comparison | `output/tables/model_comparison.csv` |